# 02. Exploratory Data Analysis & Family Screening (M2)

**Owner:** Member 2 — Statistical Research & Panel Inference  
**Objective:** Analyze all 33 product families in Favorita, verify candidate screening for deep study, and inspect statistical associations with weather.

In [1]:
import sys
from pathlib import Path
REPO_ROOT = Path(r'd:/Study/PTTk/LAB').resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from src.statistics.eda import (
    compute_family_summary,
    compute_weather_correlations,
    compute_seasonality_stats,
    screen_deep_study_candidates,
)

DATA_PATH = REPO_ROOT / 'data' / 'processed' / 'curated_sales_weather.parquet'
df = pd.read_parquet(DATA_PATH)
print('Dataset loaded successfully:', df.shape)


Dataset loaded successfully: (3000888, 35)


## 1. 33-Family Sales Summary & Zero-Rate

We examine total sales share and zero-sales frequency on open store days.

In [2]:
summary = compute_family_summary(df, open_only=True)
summary.head(10)


D:\Study\PTTK\LAB\src\statistics\eda.py:31: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = data.groupby("family")["sales"]


,total_sales,sales_share,zero_rate,mean_sales,median_sales,std_sales,count
family,,,,,,,
GROCERY I,3.434627e+08,0.319903,0.000024,4108.111079,3373.000000,2761.338014,83606
BEVERAGES,2.169545e+08,0.202073,0.000012,2594.963113,1950.000000,2291.398421,83606
PRODUCE,1.227047e+08,0.114288,0.220738,1467.654052,557.746000,2241.922446,83606
CLEANING,9.752129e+07,0.090832,0.000012,1166.438880,993.000000,690.946636,83606
DAIRY,6.448771e+07,0.060064,0.000012,771.328720,569.000000,665.691824,83606
BREAD/BAKERY,4.213395e+07,0.039244,0.000000,503.958395,436.423010,356.401970,83606
POULTRY,3.187600e+07,0.029690,0.000167,381.264556,228.915500,403.430392,83606
MEATS,3.108647e+07,0.028954,0.000012,371.821022,249.819505,463.607441,83606
PERSONAL CARE,2.459205e+07,0.022905,0.000048,294.142179,237.000000,220.979531,83606


## 2. Pre-registered Screening for Deep Study

Candidate screening rule: Zero rate <= 5%, full 22 cities presence, first sale before 2013-01-31, evaluated before 2016-08-04 (Fold F1).

In [3]:
screening = screen_deep_study_candidates(df)
screening.loc[screening['passes_screening']]


D:\Study\PTTK\LAB\src\statistics\eda.py:31: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = data.groupby("family")["sales"]


,sales_share,zero_rate,n_cities,first_sale,passes_screening
family,,,,,
GROCERY I,0.329701,0.000031,22,2013-01-01,True
BEVERAGES,0.193665,0.000016,22,2013-01-01,True
CLEANING,0.095771,0.000016,22,2013-01-01,True
DAIRY,0.060574,0.000016,22,2013-01-01,True
BREAD/BAKERY,0.040963,0.000000,22,2013-01-01,True
POULTRY,0.031602,0.000204,22,2013-01-01,True
MEATS,0.031120,0.000000,22,2013-01-01,True
PERSONAL CARE,0.023877,0.000047,22,2013-01-01,True
DELI,0.023485,0.000000,22,2013-01-01,True


## 3. Weather Correlation across Product Families

Pearson correlation between daily sales and weather measures.

In [4]:
weather_vars = [
    'precipitation_sum',
    'temperature_2m_max',
    'temperature_2m_mean',
    'relative_humidity_2m_mean',
    'sunshine_hours'
]
corrs = compute_weather_correlations(df, weather_vars)
corrs.head(10)


,precipitation_sum,temperature_2m_max,temperature_2m_mean,relative_humidity_2m_mean,sunshine_hours
AUTOMOTIVE,0.073207,-0.241600,-0.241093,-0.050912,-0.039624
BABY CARE,0.023389,0.013193,0.011221,0.027058,0.000715
BEAUTY,0.019700,-0.306905,-0.309883,-0.138745,0.012859
BEVERAGES,0.043505,-0.283865,-0.283906,-0.127618,0.002237
BOOKS,0.028028,-0.076439,-0.078377,0.030546,0.009852
BREAD/BAKERY,0.043717,-0.286442,-0.281358,-0.141783,-0.032981
CELEBRATION,0.010603,-0.141750,-0.149253,-0.075657,0.015839
CLEANING,0.069825,-0.335517,-0.334718,-0.079734,-0.017616
DAIRY,0.059163,-0.346545,-0.343756,-0.142669,-0.025747
DELI,0.069818,-0.360171,-0.364785,-0.113305,-0.019891


## 4. Key Takeaways for RQ1
1. 7 Deep-Study Families (GROCERY I, BEVERAGES, CLEANING, DAIRY, BREAD/BAKERY, PERSONAL CARE, FROZEN FOODS) capture 76% of sales with continuous historical coverage and near-zero intermittency.
2. Raw correlation with weather is modest ($|r| < 0.15$), confirming that simple bivariate correlations are masked by store-level and seasonal confounding.
3. Panel econometric specifications (Fixed Effects controlling for store x family) are required to isolate true weather associations.